# **1. Setup**



In [ ]:
!pip install transformers datasets torch scikit-learn -q

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import train_test_split

import numpy as np
from datasets import load_dataset, Dataset as HFDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# **2. Data Acquisition**



In [ ]:
dataset = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_labeled")['train']

# Subsample for faster experiments
dataset = dataset.shuffle(seed=42).select(range(500))
print("dataset from huggingface : ", dataset)

set(dataset['Category of Hallucination'])

# Convert HuggingFace dataset to list of dicts
dataset_list = [dict(example) for example in dataset]
print("dataset_list : ", dataset_list[:1])

# split
train_orig, val_orig = train_test_split(dataset_list, test_size=0.2, random_state=42)

In [ ]:
# Look at the first few examples
for i in range(4):
  print("Question: ", dataset['Question'][i])
  print("Knowledge: ", dataset['Knowledge'][i])
  print("Ground Truth: ", dataset['Ground Truth'][i])
  print("Hallucinated Answer: ", dataset['Hallucinated Answer'][i])
  print("Category of Hallucination: ", dataset['Category of Hallucination'][i])
  print()

# **3. Preprocess dataset**

In [ ]:
def preprocess(example):
    processed = []
    question = example["Question"]
    knowledge = " ".join(example["Knowledge"]) if isinstance(example["Knowledge"], list) else example["Knowledge"]
    gt_answer = example["Ground Truth"]
    hallucinated_answer = example["Hallucinated Answer"]

    if not gt_answer or not hallucinated_answer:
        return []

    premise = f"Question: {question} Context: {knowledge}"

    # Positive example (entailed)
    processed.append({"text_a": premise, "text_b": gt_answer, "label": 0})
    # Negative example (hallucinated)
    processed.append({"text_a": premise, "text_b": hallucinated_answer, "label": 1})

    return processed

# Expand train and validation sets separately
processed_train = []
for s in train_orig:
    examples = preprocess(s)   # returns a list of processed examples
    for ex in examples:
        processed_train.append(ex)

processed_val = []
for s in val_orig:
    examples = preprocess(s)
    for ex in examples:
        processed_val.append(ex)

# **4. Tokenization**

In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(checkpoint)
max_length = 256

def tokenize_dataset(processed_data):
    input_ids, attention_masks, token_type_ids, labels = [], [], [], []
    for ex in processed_data:
        encoded = tokenizer(
            ex["text_a"],
            ex["text_b"],
            truncation='longest_first',
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )
        input_ids.append(encoded['input_ids']) # tensor of shape [1, max_length]
        attention_masks.append(encoded['attention_mask']) # tensor of shape [1, max_length]
        token_type_ids.append(encoded['token_type_ids']) # tensor of shape [1, max_length]
        labels.append(torch.tensor(ex['label'])) # single scalar tensor [ ]

    # Neural networks (and DataLoader) expect tensors of shape [batch_size, ...], not a list of individual tensors.
    return (
        torch.cat(input_ids, dim=0), # shape: [num_examples, max_length]
        torch.cat(attention_masks, dim=0), # shape: [num_examples, max_length]
        torch.cat(token_type_ids, dim=0), # shape: [num_examples, max_length]
        torch.stack(labels) # shape: [num_examples]
    )

train_inputs = tokenize_dataset(processed_train)
val_inputs   = tokenize_dataset(processed_val)

# **5. Build final dataset**

In [ ]:
class HallucinationDataset(Dataset):
    def __init__(self, input_ids, attention_masks, token_type_ids, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.token_type_ids = token_type_ids
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
            "token_type_ids": self.token_type_ids[idx],
            "labels": self.labels[idx]
        }

# train_dataset, val_dataset is object of HallucinationDataset (a subclass of torch.utils.data.Dataset)
# can't directly pass the tensors to the DataLoader, PyTorch DataLoader requires a Dataset object (or anything that implements __len__ and __getitem__).
train_dataset = HallucinationDataset(*train_inputs) # unpack argument one at a time from the tuple, 4 arguments tensors (input_ids, attention_masks, token_type_ids, labels)
val_dataset   = HallucinationDataset(*val_inputs)

# len(train_dataset) returns number of examples.
# train_dataset[0] returns the first example as a dictionary.
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8)

# **6. BERT model & optimzer**

In [ ]:
# pre-defined BERT model with a classification head on top
# num_labels=2 - 0 = Faithful / Entailed, 1 = Hallucinated / Not entailed
model = BertForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5) # A variant of the Adam optimizer that decouples weight decay (regularization technique that prevents overfitting) from gradient updates.
# Learning rate is very small because BERT is already pretrained. Small lr ensures you fine-tune gently instead of destroying pretrained knowledge.

# CrossEntropyLoss Takes as input:
# - logits from the model [batch_size, num_labels]
# - labels [batch_size]
# It internally applies softmax to convert logits into probabilities and computes the negative log-likelihood.
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
for name, param in model.named_parameters():
    print(name)

# **7. Training and Evaluate**

In [ ]:
epochs = 3

print("Epoch\tTrain Loss\tVal Loss\tAcc\tF1\tPrecision\tRecall")

for epoch in range(epochs):

    # TRAIN

    # Switch the model into training mode
    # ensures layers like dropout are active, helping prevent overfitting.
    model.train()

    total_train_loss = 0

    # loop for each batch. Each batch is a dictionary of tensors. tensors that have 8 rows.
    for batch in train_loader:
        optimizer.zero_grad()
        #{
        #"input_ids": tensor([[101, 2009, ..., 102],   # sample 1
        #                     [101, 1045, ..., 102],   # sample 2
        #                     ...
        #                     [101, 2023, ..., 102]]), # sample 8
        #"attention_mask": tensor([[1,1,...,1],
        #                         [1,1,...,1],
        #                         ...
        #                         [1,1,...,1]]),
        #"token_type_ids": tensor([[0,0,...,0],
        #                         [0,0,...,0],
        #                         ...
        #                         [0,0,...,0]]),
        #"labels": tensor([0,1,0,1,0,0,1,0])
        #}
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        batch_labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=batch_labels
        )

        loss = outputs.loss # [] - tensor(0.4235, device='cuda:0', grad_fn=<NllLossBackward0>)
        logits = outputs.logits # [batch_size, num_labels] - tensor([[ 1.23, -0.56], # sample 1 [-0.45,  0.78],  # sample 2, .. , device='cuda:0', grad_fn=<AddmmBackward0>)

        # Compute gradients of the loss w.r.t. all model parameters. how each weight should change to reduce loss.
        loss.backward()
        # Optimizer (AdamW) uses the computed gradients to update weights.
        optimizer.step()

        # Accumulate training loss
        total_train_loss += loss.item()

    # Compute average training loss. Lower loss = model is learning; plateau or increasing loss = potential overfitting or learning issues.
    avg_train_loss = total_train_loss / len(train_loader)

    # VALIDATION

    # Switch the model into validation mode
    # Turns off dropout layers (so outputs are deterministic). Turns off batch norm updates (if any).
    model.eval()
    total_val_loss = 0

    # store the true labels for all validation examples.
    all_labels = []
    # store the predicted labels for all validation examples.
    all_preds = []

    with torch.no_grad(): # Prevents gradient calculations. Evaluation does not require backpropagation.
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            batch_labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=batch_labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_val_loss += loss.item()

            preds = torch.argmax(logits, dim=1) # [batch_size] - tensor([0, 1, 0, 1, 0, 1, 1, 0], device='cuda:0')

            # Collect all predictions and labels

            # moves tensors to CPU and converts to NumPy arrays and appends all items of the batch to the master list.
            all_labels.extend(batch_labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    # Compute average validation loss
    avg_val_loss = total_val_loss / len(val_loader)

    # Compute metrics (Precision, Recall, F1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary'
    )

    # Compute accuracy
    acc = np.mean(np.array(all_labels) == np.array(all_preds))

    print(f"{epoch+1}\t{avg_train_loss:.4f}\t\t{avg_val_loss:.4f}\t\t{acc:.4f}\t{f1:.4f}\t{precision:.4f}\t{recall:.4f}")

    cm = confusion_matrix(all_labels, all_preds)
    print("Confusion Matrix:\n", cm)